# Autonomous Trailer: A Knowledge-Based Agent
**Unit:** Artificial Intelligence 

This notebook develops the self-driving trailer from practical exercise 3 as a **knowledge-based agent**. The trailer must travel from **Nairobi to Kisumu** on Kenya's highway network. It stores what it knows about the road network in a **knowledge base (KB)**, and it answers questions by querying that KB.

**Step 1: Identify the Task**

The task is to help a self-driving trailer decide **which route to take from Nairobi to Kisumu**, using what it knows about the roads.

**Goal:** Deliver the cargo safely at the **lowest total cost**. Cost is driving distance in km, plus a penalty for hazards on the road. Closed roads must never be used.

**What the agent must be able to do**
- Answer questions about towns and roads, such as "is this road open?" and "what does it cost?"
- Take in **new facts** when the world changes, for example a closed road or a new hazard
- Give correct answers after the knowledge has changed

**Step 2: Assemble Relevant Knowledge**

The agent needs to know:
- Which towns exist and which county each is in
- Which towns are connected by a highway, and how far apart they are
- Whether each road is currently open or closed
- Which hazards are on each road, and how much each hazard should cost us
- The general rules for using this information

**Sources of knowledge**
| Knowledge | Source |
|---|---|
| Towns, 21 directed roads and distances | Directed graph and distance table from **practical exercise 3** (`design.md`, `UCS_Trailer.ipynb`). Distances are Google Maps driving distances rounded to the nearest km. |
| Hazard types (potholes, fog, livestock, escarpments, etc.) | The environment description in our **PEAS table** (practical exercise 3). |
| Hazard penalties | Our own design choice (explained in Step 4). |

**APIs and web scraping:** *None were used.* The knowledge base is **hard-coded** from our practical exercise 3 data, so no extra code needs to be uploaded to the repository.

**Step 3: Define the Vocabulary**

In Python, **predicates** are stored as keys in dictionaries and as rules (functions). **Constants** are the fixed values.

| Predicate | Meaning | Where it lives in the KB |
|---|---|---|
| `Town(x)` | x is a town on the network | `kb["towns"]` |
| `County(x, c)` | town x is in county c | `kb["towns"][x]["county"]` |
| `Role(x, r)` | town x is the start, goal, dead end or intermediate | `kb["towns"][x]["role"]` |
| `Road(a, b, km)` | a directed road from a to b of length km | `kb["roads"][a][b]` |
| `Status(a, b, s)` | road a to b is open or closed | `kb["status"][(a, b)]` |
| `Hazard(a, b, h)` | hazard h is present on road a to b | `kb["hazards"][(a, b)]` |
| `HazardPenalty(h, p)` | hazard h adds p km-equivalent of cost | `kb["rules"]["hazard_penalty_km"][h]` |
| `Usable(a, b)` | road exists **and** is open (a derived rule) | function `is_reachable_road(a, b)` |
| `EffectiveCost(a, b, c)` | km + hazard penalties (a derived rule) | function `get_neighbors(a)` |

**Constants**
- **Towns (19):** Nairobi, Ruiru, Athi River, Naivasha, Narok, Thika, Karatina, Nyeri, Nyahururu, Nakuru, Makindu, Kajiado, Gilgil, Kericho, Kisumu, Bomet, Kisii, Migori, Homa Bay
- **Start:** Nairobi | **Goal:** Kisumu
- **Status values:** open, closed
- **Roles:** start, goal, dead_end, intermediate

**Step 4: Encode General Knowledge about the Domain** `[3 marks]`

The knowledge base is a single Python dictionary called `kb` with five sections: `towns`, `roads`, `status`, `hazards` and `rules`. The cells below create the raw data, build `kb` from it, add query helpers, and then check that the KB is correct.

In [11]:
# ------------------------------------------------------------
# STEP 4a: RAW DATA (taken from practical exercise 3)
# ------------------------------------------------------------

# TOWNS: every town on the network (19 in total).
# Each entry is  town -> (county, role)
# "role" says what the town means for our problem:
#   "start"        -> where the trailer begins
#   "goal"         -> where the trailer must arrive
#   "dead_end"     -> a town with no onward road in our graph (cannot lead to Kisumu)
#   "intermediate" -> any other town on the way
TOWNS = {
    "Nairobi":    ("Nairobi",   "start"),
    "Ruiru":      ("Kiambu",    "intermediate"),
    "Athi River": ("Machakos",  "intermediate"),
    "Naivasha":   ("Nakuru",    "intermediate"),
    "Narok":      ("Narok",     "intermediate"),
    "Thika":      ("Kiambu",    "intermediate"),
    "Karatina":   ("Nyeri",     "intermediate"),
    "Nyeri":      ("Nyeri",     "intermediate"),
    "Nyahururu":  ("Laikipia",  "intermediate"),
    "Nakuru":     ("Nakuru",    "intermediate"),
    "Makindu":    ("Makueni",   "dead_end"),
    "Kajiado":    ("Kajiado",   "dead_end"),
    "Gilgil":     ("Nakuru",    "intermediate"),
    "Kericho":    ("Kericho",   "intermediate"),
    "Kisumu":     ("Kisumu",    "goal"),
    "Bomet":      ("Bomet",     "intermediate"),
    "Kisii":      ("Kisii",     "intermediate"),
    "Migori":     ("Migori",    "dead_end"),
    "Homa Bay":   ("Homa Bay",  "intermediate"),
}

# EDGES: the 21 directed roads, as (from, to, distance in km).
# These are exactly the 21 rows of the distance table in practical exercise 3.
EDGES = [
    ("Nairobi",    "Ruiru",       25),
    ("Nairobi",    "Athi River",  30),
    ("Nairobi",    "Naivasha",    91),
    ("Nairobi",    "Narok",      143),
    ("Ruiru",      "Thika",       21),
    ("Thika",      "Karatina",    88),
    ("Karatina",   "Nyeri",       25),
    ("Nyeri",      "Nyahururu",  104),
    ("Nyahururu",  "Nakuru",      68),
    ("Athi River", "Makindu",    149),
    ("Athi River", "Kajiado",     50),
    ("Naivasha",   "Gilgil",      30),
    ("Gilgil",     "Nakuru",      42),
    ("Nakuru",     "Kericho",    107),
    ("Kericho",    "Kisumu",      84),
    ("Narok",      "Bomet",       81),
    ("Bomet",      "Kisii",       82),
    ("Kisii",      "Kisumu",     114),
    ("Kisii",      "Migori",      67),
    ("Kisii",      "Homa Bay",    52),
    ("Homa Bay",   "Kisumu",     108),
]

print(f"Raw data loaded: {len(TOWNS)} towns, {len(EDGES)} roads")

Raw data loaded: 19 towns, 21 roads


In [12]:
# ------------------------------------------------------------
# STEP 4b: BUILD THE KNOWLEDGE BASE
# ------------------------------------------------------------
# The KB is ONE dictionary with five sections:
#   "towns"   : town -> {"county": ..., "role": ...}
#   "roads"   : from_town -> {to_town: distance_km}     (directed roads)
#   "status"  : (from_town, to_town) -> "open" or "closed"
#   "hazards" : (from_town, to_town) -> [list of hazard names]  (empty at the start)
#   "rules"   : general rules that apply to the whole domain

kb = {
    "towns":   {},
    "roads":   {},
    "status":  {},
    "hazards": {},      # no hazards are known yet; we add them in Step 5

    "rules": {
        # The only two values a road's status may take.
        "valid_status": ["open", "closed"],

        # RULE: each hazard adds a penalty to the cost of a road.
        # Penalties are in "km-equivalent": a road with fog costs
        # its distance PLUS 10. These numbers are our own design choice;
        # a higher number means the hazard is more dangerous/slow.
        # The hazard types come from the environment in our PEAS table.
        "hazard_penalty_km": {
            "pothole":            5,
            "fog":               10,
            "livestock crossing": 10,
            "steep escarpment":  10,
            "heavy traffic":     15,
            "roadworks":         20,
            "flooding":          40,
        },

        # RULES written in words, to document how the helper functions behave.
        "usable_road":    "A road can be used only if it exists AND its status is 'open'.",
        "effective_cost": "Effective cost of a road = distance in km + the penalties of all hazards on it.",
        "dead_end":       "A dead-end town has no outgoing roads, so it can never lead on to the goal.",
    },
}

# Fill in the towns. Every town also gets an (empty) road dictionary,
# so that dead-end towns still exist as keys in kb["roads"].
for town, (county, role) in TOWNS.items():
    kb["towns"][town] = {"county": county, "role": role}
    kb["roads"][town] = {}

# Fill in the roads. Every road starts out "open".
for from_town, to_town, km in EDGES:
    kb["roads"][from_town][to_town] = km
    kb["status"][(from_town, to_town)] = "open"

print("Knowledge base built.")
print("Sections:", list(kb.keys()))

Knowledge base built.
Sections: ['towns', 'roads', 'status', 'hazards', 'rules']


In [13]:
# ------------------------------------------------------------
# STEP 4c: HELPER FUNCTIONS 
# These only READ the KB. They never change it.
# ------------------------------------------------------------

def normalise(name):
    """Clean up a town name so 'nairobi ' and 'NAIROBI' both become 'Nairobi'."""
    return name.strip().title()


def town_exists(town):
    """True if the town is in the knowledge base."""
    return normalise(town) in kb["towns"]


def road_exists(a, b):
    """True if there is a directed road from town a to town b (open OR closed)."""
    a, b = normalise(a), normalise(b)
    return b in kb["roads"].get(a, {})


def is_reachable_road(a, b):
    """
    RULE: Usable(a, b)
    True only if the road exists AND it is currently open.
    """
    a, b = normalise(a), normalise(b)
    return road_exists(a, b) and kb["status"].get((a, b)) == "open"


def hazard_penalty(a, b):
    """Total penalty (in km-equivalent) of all hazards currently on the road a -> b."""
    a, b = normalise(a), normalise(b)
    penalties = kb["rules"]["hazard_penalty_km"]
    hazards_on_road = kb["hazards"].get((a, b), [])   # [] if there are none
    return sum(penalties[h] for h in hazards_on_road)


def get_neighbors(town):
    """
    The MAIN function for Person B's search.
    Returns a list of (next_town, effective_cost) for every OPEN road leaving `town`.
        effective_cost = distance_km + hazard_penalty
    - Closed roads are left out.
    - Dead-end or unknown towns return an empty list [].
    """
    town = normalise(town)
    neighbors = []
    for next_town, km in kb["roads"].get(town, {}).items():
        if is_reachable_road(town, next_town):
            neighbors.append((next_town, km + hazard_penalty(town, next_town)))
    return neighbors


def route_cost(route):
    """
    Total effective cost of a route given as a list of towns,
    e.g. ["Nairobi", "Naivasha", "Gilgil"].
    Returns None if any leg of the route is missing or closed.
    """
    total = 0
    for a, b in zip(route, route[1:]):                # look at each pair of neighbours
        costs_from_a = dict(get_neighbors(a))         # {next_town: cost}
        if normalise(b) not in costs_from_a:
            return None                               # this leg cannot be driven
        total += costs_from_a[normalise(b)]
    return total


def show_kb():
    """Print every road in the KB with its distance, status and hazards."""
    print("\n--- Current Knowledge Base ---")
    for town, roads_out in kb["roads"].items():
        if not roads_out:
            print(f"{town:11} -> dead end (no outgoing roads)")
            continue
        for next_town, km in roads_out.items():
            status = kb["status"][(town, next_town)]
            hazards = kb["hazards"].get((town, next_town), [])
            hazard_text = f" | hazards: {', '.join(hazards)}" if hazards else ""
            print(f"{town:11} -> {next_town:11} {km:4} km  [{status}]{hazard_text}")
    print("------------------------------")

In [14]:
# ------------------------------------------------------------
# STEP 4d: VIEW THE KB AND CHECK IT IS CORRECT
# ------------------------------------------------------------
show_kb()

# --- Sanity checks: if any of these fail, Python shows an AssertionError ---
total_roads = sum(len(r) for r in kb["roads"].values())
assert len(kb["towns"]) == 19, "Expected 19 towns"
assert total_roads == 21, "Expected 21 roads"

# Every road must connect two towns that exist.
for a, roads_out in kb["roads"].items():
    for b in roads_out:
        assert a in kb["towns"] and b in kb["towns"], f"Unknown town in road {a} -> {b}"

# Dead-end towns must have no outgoing roads.
for town, info in kb["towns"].items():
    if info["role"] == "dead_end":
        assert kb["roads"][town] == {}, f"{town} is a dead end but has roads"

# The best route found in practical exercise 3 must still cost 354 km.
KNOWN_BEST_ROUTE = ["Nairobi", "Naivasha", "Gilgil", "Nakuru", "Kericho", "Kisumu"]
assert route_cost(KNOWN_BEST_ROUTE) == 354, "Route cost does not match practical exercise 3"

print(f"\nAll checks passed: {len(kb['towns'])} towns, {total_roads} roads.")
print("Cost of the known best route:", route_cost(KNOWN_BEST_ROUTE), "km")


--- Current Knowledge Base ---
Nairobi     -> Ruiru         25 km  [open]
Nairobi     -> Athi River    30 km  [open]
Nairobi     -> Naivasha      91 km  [open]
Nairobi     -> Narok        143 km  [open]
Ruiru       -> Thika         21 km  [open]
Athi River  -> Makindu      149 km  [open]
Athi River  -> Kajiado       50 km  [open]
Naivasha    -> Gilgil        30 km  [open]
Narok       -> Bomet         81 km  [open]
Thika       -> Karatina      88 km  [open]
Karatina    -> Nyeri         25 km  [open]
Nyeri       -> Nyahururu    104 km  [open]
Nyahururu   -> Nakuru        68 km  [open]
Nakuru      -> Kericho      107 km  [open]
Makindu     -> dead end (no outgoing roads)
Kajiado     -> dead end (no outgoing roads)
Gilgil      -> Nakuru        42 km  [open]
Kericho     -> Kisumu        84 km  [open]
Kisumu      -> dead end (no outgoing roads)
Bomet       -> Kisii         82 km  [open]
Kisii       -> Kisumu       114 km  [open]
Kisii       -> Migori        67 km  [open]
Kisii       -> Homa

**Step 5: Add New Facts to the Knowledge Base** `[4 marks]`

A knowledge-based agent must be able to **learn new facts** while it is running. We provide functions to change each part of the KB:

| Function | What it does |
|---|---|
| `add_town(name, county)` | Adds a new town |
| `remove_town(name)` | Removes a town and every road into or out of it |
| `add_road(a, b, km)` | Adds a new directed road (starts open) |
| `remove_road(a, b)` | Deletes a road |
| `update_distance(a, b, km)` | Corrects a distance |
| `set_status(a, b, status)` | Opens or closes a road |
| `add_hazard_type(name, penalty)` | Teaches the agent a new kind of hazard (a new **rule**) |
| `add_hazard(a, b, hazard)` | Records a hazard on a road |
| `clear_hazards(a, b)` | Removes all hazards from a road |

Each function checks its input first, so a bad fact cannot corrupt the KB. Each prints a message and returns `True` (success) or `False` (rejected).

In [15]:
# ------------------------------------------------------------
# STEP 5a: ADDING / REMOVING TOWNS AND ROADS
# ------------------------------------------------------------

def add_town(name, county, role="intermediate"):
    """Add a new town to the KB. It starts with no roads."""
    name = normalise(name)
    if town_exists(name):
        print(f"Rejected: town '{name}' already exists.")
        return False
    kb["towns"][name] = {"county": county, "role": role}
    kb["roads"][name] = {}                 # a new town starts with no outgoing roads
    print(f"Added town: {name} ({county} County)")
    return True


def add_road(a, b, km):
    """Add a new directed road a -> b. It starts out 'open'."""
    a, b = normalise(a), normalise(b)
    if not (town_exists(a) and town_exists(b)):
        print(f"Rejected: both '{a}' and '{b}' must already be towns in the KB (use add_town first).")
        return False
    if road_exists(a, b):
        print(f"Rejected: road {a} -> {b} already exists (use update_distance to change it).")
        return False
    if not isinstance(km, (int, float)) or km <= 0:
        print("Rejected: distance must be a positive number.")
        return False
    kb["roads"][a][b] = km
    kb["status"][(a, b)] = "open"
    print(f"Added road: {a} -> {b} ({km} km, open)")
    return True


def remove_road(a, b, verbose=True):
    """Delete the road a -> b, together with its status and hazards."""
    a, b = normalise(a), normalise(b)
    if not road_exists(a, b):
        if verbose:
            print(f"Rejected: no road {a} -> {b} to remove.")
        return False
    del kb["roads"][a][b]
    kb["status"].pop((a, b), None)         # pop(..., None) = delete if present, no error if not
    kb["hazards"].pop((a, b), None)
    if verbose:
        print(f"Removed road: {a} -> {b}")
    return True


def remove_town(name):
    """Remove a town and every road going into or out of it."""
    name = normalise(name)
    if not town_exists(name):
        print(f"Rejected: town '{name}' does not exist.")
        return False
    if kb["towns"][name]["role"] in ("start", "goal"):
        print(f"Rejected: '{name}' is the start or goal town and cannot be removed.")
        return False
    for other in list(kb["roads"]):                       # roads INTO the town
        remove_road(other, name, verbose=False)
    for next_town in list(kb["roads"][name]):             # roads OUT of the town
        remove_road(name, next_town, verbose=False)
    del kb["roads"][name]
    del kb["towns"][name]
    print(f"Removed town: {name} (and all its roads)")
    return True


def update_distance(a, b, km):
    """Correct the distance of an existing road."""
    a, b = normalise(a), normalise(b)
    if not road_exists(a, b):
        print(f"Rejected: no road {a} -> {b} to update.")
        return False
    if not isinstance(km, (int, float)) or km <= 0:
        print("Rejected: distance must be a positive number.")
        return False
    old = kb["roads"][a][b]
    kb["roads"][a][b] = km
    print(f"Updated distance {a} -> {b}: {old} km -> {km} km")
    return True

In [16]:
# ------------------------------------------------------------
# STEP 5b: ROAD STATUS AND HAZARDS
# ------------------------------------------------------------

def set_status(a, b, status):
    """Open or close a road. Only the values in kb['rules']['valid_status'] are allowed."""
    a, b = normalise(a), normalise(b)
    status = status.strip().lower()
    if not road_exists(a, b):
        print(f"Rejected: no road {a} -> {b}.")
        return False
    if status not in kb["rules"]["valid_status"]:
        print(f"Rejected: status must be one of {kb['rules']['valid_status']}.")
        return False
    kb["status"][(a, b)] = status
    print(f"Road {a} -> {b} is now {status.upper()}")
    return True


def add_hazard_type(name, penalty_km):
    """Teach the agent a NEW kind of hazard (this adds a new rule to kb['rules'])."""
    name = name.strip().lower()
    if name in kb["rules"]["hazard_penalty_km"]:
        print(f"Rejected: hazard type '{name}' is already known.")
        return False
    if not isinstance(penalty_km, (int, float)) or penalty_km < 0:
        print("Rejected: penalty must be a number that is 0 or more.")
        return False
    kb["rules"]["hazard_penalty_km"][name] = penalty_km
    print(f"Added hazard type: '{name}' (penalty {penalty_km} km)")
    return True


def add_hazard(a, b, hazard):
    """Record that a known hazard is present on the road a -> b."""
    a, b = normalise(a), normalise(b)
    hazard = hazard.strip().lower()
    if not road_exists(a, b):
        print(f"Rejected: no road {a} -> {b}.")
        return False
    if hazard not in kb["rules"]["hazard_penalty_km"]:
        print(f"Rejected: unknown hazard '{hazard}'. Known: {list(kb['rules']['hazard_penalty_km'])}")
        return False
    current = kb["hazards"].setdefault((a, b), [])        # create an empty list if none yet
    if hazard in current:
        print(f"Rejected: '{hazard}' is already recorded on {a} -> {b}.")
        return False
    current.append(hazard)
    print(f"Added hazard '{hazard}' on {a} -> {b}")
    return True


def clear_hazards(a, b):
    """Remove all hazards from the road a -> b."""
    a, b = normalise(a), normalise(b)
    if not road_exists(a, b):
        print(f"Rejected: no road {a} -> {b}.")
        return False
    kb["hazards"].pop((a, b), None)
    print(f"Cleared all hazards on {a} -> {b}")
    return True

In [17]:
# ------------------------------------------------------------
# STEP 5c: USER-FRIENDLY VERSION (asks the user to type the new fact)
# ------------------------------------------------------------
# This is defined but NOT called, because input() pauses the notebook and
# would stop "Run all" from finishing. To try it, remove the # on the last line.

def add_road_interactive():
    """Ask the user for a new road and add it to the KB."""
    a  = input("From which town? ")
    b  = input("To which town? ")
    try:
        km = float(input("Distance in km? "))
    except ValueError:
        print("Rejected: distance must be a number.")
        return False
    return add_road(a, b, km)

# add_road_interactive()

In [18]:
# ------------------------------------------------------------
# STEP 5d: DEMONSTRATION - add new facts and see how answers change
# At the end we RESTORE the KB so the next person (person B) starts from the clean original.
# ------------------------------------------------------------

print("=== 5a. NEW TOWN AND ROAD ===")
print("Roads out of Nakuru BEFORE:", get_neighbors("Nakuru"))
add_town("Eldoret", "Uasin Gishu")
add_road("Nakuru", "Eldoret", 155)            # illustrative distance
print("Roads out of Nakuru AFTER: ", get_neighbors("Nakuru"))

print("\n=== 5b. CLOSE A ROAD ===")
print("Cost of best known route BEFORE:", route_cost(KNOWN_BEST_ROUTE), "km")
set_status("Naivasha", "Gilgil", "closed")
print("Roads out of Naivasha now:      ", get_neighbors("Naivasha"))
print("Cost of best known route AFTER: ", route_cost(KNOWN_BEST_ROUTE), "(None = route blocked)")
set_status("Naivasha", "Gilgil", "open")      # reopen it

print("\n=== 5c. NEW HAZARD ===")
add_hazard("Nakuru", "Kericho", "fog")
print("Roads out of Nakuru with fog:", get_neighbors("Nakuru"))
print("Cost of best known route with fog:", route_cost(KNOWN_BEST_ROUTE), "km (354 + 10)")

print("\n=== 5d. NEW HAZARD TYPE (a new rule) ===")
add_hazard_type("landslide", 30)              # this one stays in the KB
add_hazard("Narok", "Bomet", "landslide")
print("Roads out of Narok:", get_neighbors("Narok"), "(81 + 30)")

print("\n=== 5e. REJECTED (INVALID) FACTS ===")
add_road("Nairobi", "Mombasa", 480)           # Mombasa is not in the KB
set_status("Nairobi", "Ruiru", "maybe")       # not a valid status
add_hazard("Nairobi", "Ruiru", "tornado")     # unknown hazard type
remove_town("Kisumu")                         # goal town is protected

# --- RESTORE the original KB so the demo leaves no side effects ---
print("\n=== RESTORING THE ORIGINAL KB ===")
remove_town("Eldoret")
clear_hazards("Nakuru", "Kericho")
clear_hazards("Narok", "Bomet")

assert route_cost(KNOWN_BEST_ROUTE) == 354
assert sum(len(r) for r in kb["roads"].values()) == 21
assert len(kb["towns"]) == 19
print("KB restored: 19 towns, 21 roads, best route costs 354 km.")

=== 5a. NEW TOWN AND ROAD ===
Roads out of Nakuru BEFORE: [('Kericho', 107)]
Added town: Eldoret (Uasin Gishu County)
Added road: Nakuru -> Eldoret (155 km, open)
Roads out of Nakuru AFTER:  [('Kericho', 107), ('Eldoret', 155)]

=== 5b. CLOSE A ROAD ===
Cost of best known route BEFORE: 354 km
Road Naivasha -> Gilgil is now CLOSED
Roads out of Naivasha now:       []
Cost of best known route AFTER:  None (None = route blocked)
Road Naivasha -> Gilgil is now OPEN

=== 5c. NEW HAZARD ===
Added hazard 'fog' on Nakuru -> Kericho
Roads out of Nakuru with fog: [('Kericho', 117), ('Eldoret', 155)]
Cost of best known route with fog: 364 km (354 + 10)

=== 5d. NEW HAZARD TYPE (a new rule) ===
Added hazard type: 'landslide' (penalty 30 km)
Added hazard 'landslide' on Narok -> Bomet
Roads out of Narok: [('Bomet', 111)] (81 + 30)

=== 5e. REJECTED (INVALID) FACTS ===
Rejected: both 'Nairobi' and 'Mombasa' must already be towns in the KB (use add_town first).
Rejected: status must be one of ['open', 